# 🇪🇬 Egyptian ID OCR — Notebook: Two-Stage Detection with NASO7Y Models

This notebook demonstrates the **two-stage YOLO detection pipeline** using NASO7Y pre-trained models with automatic class mapping translation.

**Pipeline:**
```
Full Image → Card Detection → Card Crop → Field Detection → Field Crops → OCR
```

**Input**: Raw ID card images  
**Output**: Cropped field images ready for OCR

## 1. Setup Environment


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Image as IPImage

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

print(f"📂 Project root: {ROOT}")

## 2. Load Detection Models


In [ ]:
from src.card_detector import CardDetector, load_card_detector

# Initialize two-stage detector
detector = load_card_detector(
    card_model_path=str(ROOT / "weights" / "card_detection.pt"),
    field_model_path=str(ROOT / "weights" / "field_detection.pt"),
)

print("✅ Two-stage detector loaded successfully!")
print()
print("📋 Model classes:")
print("   Card Detection: 8 corner/edge classes")
print("   Field Detection: 31 classes (12 valid for OCR)")

## 3. Test on Single Image


In [ ]:
# Load a test image
test_image_path = ROOT / "train" / "images" / "006cc843-52e3-48ab-958f-59bf42c108fd_png.rf.1392ac8180bd85b549b03f2d8528da26.jpg"
image = cv2.imread(str(test_image_path))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f"📷 Input image: {image.shape[1]}x{image.shape[0]}")

# Display original image
plt.figure(figsize=(8, 8))
plt.imshow(image_rgb)
plt.title("Original ID Card Image")
plt.axis('off')
plt.tight_layout()
plt.show()

### 3.1 Run Two-Stage Detection (NASO7Y Format)


In [ ]:
# Run detection - NASO7Y format
card_crop, fields = detector.detect_full(image, translate_to_project=False)

print(f"✂️  Card crop: {card_crop.shape[1]}x{card_crop.shape[0]}")
print(f"📋 Fields detected: {len(fields)}")
print()
print("Fields (NASO7Y format):")
for field_name, (crop, conf) in fields.items():
    print(f"   - {field_name:15s}: {crop.shape[1]}x{crop.shape[0]} (conf: {conf:.2f})")

In [ ]:
# Display card crop and detected fields
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

# Card crop
card_rgb = cv2.cvtColor(card_crop, cv2.COLOR_BGR2RGB)
axes[0].imshow(card_rgb)
axes[0].set_title(f"Card Crop\n{card_crop.shape[1]}x{card_crop.shape[0]}", fontsize=12)
axes[0].axis('off')

# Field crops
for idx, (field_name, (crop, conf)) in enumerate(fields.items(), start=1):
    if idx > 8:  # Limit to 8 fields
        break
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(crop_rgb)
    axes[idx].set_title(f"{field_name}\n{crop.shape[1]}x{crop.shape[0]} (conf: {conf:.2f})", fontsize=10)
    axes[idx].axis('off')

# Hide remaining axes
for idx in range(len(fields) + 1, 9):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 3.2 Run Two-Stage Detection (Project Format - Translated)


In [ ]:
# Run detection - Project format (translated)
card_crop, fields = detector.detect_full(image, translate_to_project=True)

print(f"✂️  Card crop: {card_crop.shape[1]}x{card_crop.shape[0]}")
print(f"📋 Fields detected: {len(fields)}")
print()
print("Fields (Project format - translated):")
for field_name, (crop, conf) in fields.items():
    print(f"   - {field_name:15s}: {crop.shape[1]}x{crop.shape[0]} (conf: {conf:.2f})")

In [ ]:
# Display translated field crops
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (field_name, (crop, conf)) in enumerate(fields.items()):
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(crop_rgb)
    axes[idx].set_title(f"{field_name}\n{crop.shape[1]}x{crop.shape[0]} (conf: {conf:.2f})", fontsize=11)
    axes[idx].axis('off')

# Hide remaining axes
for idx in range(len(fields), 6):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 3.3 Get Class Mapping


In [ ]:
# Get both formats with mapping
card_crop, fields, mapping = detector.detect_full_with_mapping(image)

print("🔄 Class Mapping (NASO7Y → Project):")
print("=" * 50)
for naso7y_name, project_name in mapping.items():
    print(f"   {naso7y_name:15s} → {project_name}")

## 4. Process Multiple Images


In [ ]:
# Get list of test images
images_dir = ROOT / "train" / "images"
image_files = sorted(list(images_dir.glob("*.jpg"))[:10])  # First 10 images

print(f"📋 Processing {len(image_files)} images...")

In [ ]:
# Process all images
results = []

for img_path in tqdm(image_files, desc="Processing images"):
    try:
        # Load image
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        
        # Run detection
        card_crop, fields = detector.detect_full(image, translate_to_project=True)
        
        # Store results
        for field_name, (crop, conf) in fields.items():
            results.append({
                'image': img_path.name,
                'field': field_name,
                'width': crop.shape[1],
                'height': crop.shape[0],
                'confidence': round(conf, 3),
            })
    except Exception as e:
        print(f"⚠️  Error processing {img_path.name}: {e}")

# Create DataFrame
df_results = pd.DataFrame(results)
print(f"\n✅ Processed {len(df_results['image'].unique())} images")
print(f"📊 Total fields detected: {len(df_results)}")

In [ ]:
# Display statistics
print("\n📊 Detection Statistics:")
print("=" * 50)
print(f"\nFields per image: {df_results.groupby('image').size().describe().to_dict()}")
print(f"\nAverage confidence by field:")
print(df_results.groupby('field')['confidence'].mean().sort_values(ascending=False).round(3))

## 5. Save Cropped Fields


In [ ]:
# Save cropped fields for OCR
output_dir = ROOT / "rec" / "images" / "two_stage"
output_dir.mkdir(parents=True, exist_ok=True)

saved_count = 0
for img_path in tqdm(image_files[:5], desc="Saving crops"):  # First 5 images
    try:
        image = cv2.imread(str(img_path))
        card_crop, fields = detector.detect_full(image, translate_to_project=True)
        
        for field_name, (crop, conf) in fields.items():
            save_name = f"{img_path.stem}_{field_name}.jpg"
            save_path = output_dir / save_name
            cv2.imwrite(str(save_path), crop)
            saved_count += 1
    except Exception as e:
        print(f"⚠️  Error: {e}")

print(f"\n✅ Saved {saved_count} field crops to: {output_dir}")

## 6. Export Metadata CSV


In [ ]:
# Create metadata DataFrame
metadata = []
for img_path in tqdm(image_files[:5], desc="Creating metadata"):
    try:
        image = cv2.imread(str(img_path))
        card_crop, fields = detector.detect_full(image, translate_to_project=True)
        
        for field_name, (crop, conf) in fields.items():
            save_name = f"{img_path.stem}_{field_name}.jpg"
            metadata.append({
                'image_path': f"rec/images/two_stage/{save_name}",
                'field': field_name,
                'split': 'train',
                'orig_image': img_path.name,
                'confidence': round(conf, 3),
                'label_text': '',  # To be filled by OCR
            })
    except Exception as e:
        print(f"⚠️  Error: {e}")

# Save to CSV
df_metadata = pd.DataFrame(metadata)
metadata_path = ROOT / "crops_metadata_two_stage.csv"
df_metadata.to_csv(metadata_path, index=False, encoding='utf-8-sig')

print(f"✅ Metadata saved to: {metadata_path}")
print(f"📊 Total records: {len(df_metadata)}")
print(f"\nPreview:")
display(df_metadata.head(10))

## 7. Compare: NASO7Y vs Project Format


In [ ]:
from src.class_mapping import get_naso7y_valid_classes, get_project_valid_classes

print("NASO7Y Valid Classes:")
print("=" * 50)
for class_id, class_name in get_naso7y_valid_classes().items():
    print(f"   {class_id:2d}: {class_name}")

print("\n\nProject Valid Classes:")
print("=" * 50)
for class_id, class_name in get_project_valid_classes().items():
    print(f"   {class_id:2d}: {class_name}")

## 8. Summary


In [ ]:
print("\n" + "=" * 60)
print("📊 TWO-STAGE DETECTION SUMMARY")
print("=" * 60)
print()
print("✅ Models Loaded:")
print("   - Card Detection: weights/card_detection.pt")
print("   - Field Detection: weights/field_detection.pt")
print()
print("✅ Images Processed:", len(df_results['image'].unique()))
print("✅ Fields Detected:", len(df_results))
print("✅ Crops Saved:", saved_count)
print()
print("📋 Field Distribution:")
field_counts = df_results['field'].value_counts()
for field, count in field_counts.items():
    print(f"   - {field:20s}: {count}")
print()
print("📂 Output:")
print(f"   - Crops: {output_dir}")
print(f"   - Metadata: {metadata_path}")
print()
print("🚀 Next Steps:")
print("   1. Review cropped fields")
print("   2. Run OCR: python scripts/label_crops.py --method qari")
print("   3. Train PaddleOCR model")
print("=" * 60)